# CNN from Scratch - Demo Notebook

This notebook demonstrates training and evaluating a custom CNN on CIFAR-10.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MAYANK12-WQ/CNN-from-Scratch/blob/main/demo.ipynb)

## Setup

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
# If running on Colab, clone the repository
# !git clone https://github.com/MAYANK12-WQ/CNN-from-Scratch.git
# %cd CNN-from-Scratch

In [ ]:
# Install dependencies (if needed)
# !pip install -r requirements.txt

## Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from model import CustomCNN
from dataset import get_data_loaders, denormalize
from utils import (
    visualize_filters,
    visualize_feature_maps,
    plot_training_history,
    compute_gradcam
)

## Load Dataset

In [ ]:
# Load CIFAR-10 dataset
train_loader, test_loader, classes = get_data_loaders(batch_size=128)

print(f"Classes: {classes}")

## Visualize Sample Data

In [ ]:
# Get a batch of training data
dataiter = iter(train_loader)
images, labels = next(dataiter)

# Denormalize images for visualization
images_denorm = denormalize(images)

# Plot sample images
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('Sample CIFAR-10 Images', fontsize=16, fontweight='bold')

for idx in range(16):
    ax = axes[idx // 8, idx % 8]
    img = images_denorm[idx].permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.set_title(classes[labels[idx]], fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Initialize Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = CustomCNN(num_classes=10)
model = model.to(device)

print(f"\nTotal parameters: {model.get_num_parameters():,}")
print(f"\nModel architecture:")
print(model)

## Training (Quick Demo - 5 Epochs)

For full training (50 epochs), use `train.py` script.

In [ ]:
# Training configuration
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
num_epochs = 5

# Training loop
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    train_loss = train_loss / total
    train_acc = 100. * correct / total
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    val_loss = val_loss / total
    val_acc = 100. * correct / total
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

print("\nTraining completed!")

## Visualize Training Progress

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
ax1.plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
ax1.plot(epochs, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy plot
ax2.plot(epochs, history['train_acc'], 'b-', label='Train Acc', linewidth=2)
ax2.plot(epochs, history['val_acc'], 'r-', label='Val Acc', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Visualize Learned Filters

In [ ]:
# Visualize filters from first convolutional layer
visualize_filters(model, layer_name='conv1', num_filters=64)

## Visualize Feature Maps

In [ ]:
# Get a sample image
dataiter = iter(test_loader)
images, labels = next(dataiter)
sample_image = images[0].to(device)

# Show original image
img_denorm = denormalize(sample_image.cpu()).permute(1, 2, 0).numpy()
img_denorm = np.clip(img_denorm, 0, 1)

plt.figure(figsize=(4, 4))
plt.imshow(img_denorm)
plt.title(f'Original Image: {classes[labels[0]]}')
plt.axis('off')
plt.show()

# Visualize feature maps from different layers
for layer_name in ['conv1', 'conv2', 'conv3']:
    visualize_feature_maps(model, sample_image, layer_name=layer_name, num_maps=16)

## Test Predictions

In [ ]:
# Make predictions on test samples
model.eval()
dataiter = iter(test_loader)
images, labels = next(dataiter)

images_gpu = images.to(device)
with torch.no_grad():
    outputs = model(images_gpu)
    _, predicted = outputs.max(1)

# Visualize predictions
images_denorm = denormalize(images)

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle('Model Predictions', fontsize=16, fontweight='bold')

for idx in range(16):
    ax = axes[idx // 8, idx % 8]
    img = images_denorm[idx].permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    
    true_label = classes[labels[idx]]
    pred_label = classes[predicted[idx]]
    color = 'green' if labels[idx] == predicted[idx] else 'red'
    
    ax.set_title(f'True: {true_label}\nPred: {pred_label}', fontsize=8, color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Evaluate Model Performance

In [ ]:
# Evaluate on entire test set
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Evaluating'):
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# Calculate per-class accuracy
from sklearn.metrics import classification_report

print("\nClassification Report:")
print("=" * 60)
print(classification_report(all_labels, all_preds, target_names=classes))

## Save Model

In [ ]:
# Save trained model
torch.save({
    'model_state_dict': model.state_dict(),
    'val_acc': history['val_acc'][-1],
}, 'cnn_model.pth')

print("Model saved as 'cnn_model.pth'")

## Conclusion

This notebook demonstrated:
- Custom CNN architecture implementation
- Training on CIFAR-10 dataset
- Visualizing learned features
- Model evaluation

For full training (50 epochs) and better accuracy (~85%), use the `train.py` script with GPU acceleration.